In [1]:
import torch
import torch.nn as nn
import copy
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")


/Users/sambhav/Documents/vscode_projects/cs329h-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


In [2]:
# 2. Model Initialization

model_name = "gpt2"  # 124M params
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# pi_ref
pi_ref = GPT2LMHeadModel.from_pretrained(model_name).to(device)
pi_ref.eval()  # ref policy is frozen

# pi_star (Oracle)
# Create pi_star by adding Gaussian noise to pi_ref weights
sigma = 0.01  # Noise scale - configurable
pi_star = copy.deepcopy(pi_ref)

with torch.no_grad():
    for param in pi_star.parameters():
        noise = torch.randn_like(param) * sigma
        param.add_(noise)
pi_star.eval()

# pi_train (Policy to optimize)
# We will have two copies for the two experiments
pi_train_baseline = copy.deepcopy(pi_ref)
pi_train_graph = copy.deepcopy(pi_ref)

print("Models initialized: pi_ref, pi_star, pi_train_baseline, pi_train_graph")

# 3. Data Generation


def generate_completions(model, tokenizer, prompt, n=100, max_length=20):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    # We replicate the input batch to generate n sequences in parallel (or chunks if n is large)

    completions = []

    # Generate in batches of 10 to be safe
    batch_size = 10
    num_batches = n // batch_size

    with torch.no_grad():
        for _ in tqdm(range(num_batches), desc="Generating completions"):
            # repeat inputs
            batch_inputs = {k: v.repeat(batch_size, 1) for k, v in inputs.items()}
            outputs = model.generate(
                **batch_inputs,
                max_length=max_length,
                do_sample=True,
                top_k=50,
                pad_token_id=tokenizer.eos_token_id,
            )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            completions.extend(decoded)

    return completions


def get_log_probs(model, tokenizer, completions):
    # Returns length-normalized log-likelihoods
    scores = []

    with torch.no_grad():
        for text in tqdm(completions, desc="Scoring"):
            inputs = tokenizer(text, return_tensors="pt").to(device)
            input_ids = inputs["input_ids"]
            labels = input_ids.clone()

            outputs = model(input_ids, labels=labels)
            loss = outputs.loss  # This is cross entropy (avg neg log likelihood)
            # log_likelihood = -loss * sequence_length
            # normalized = -loss

            # We want higher is better.
            # loss is minimized, so -loss is maximized.
            scores.append(-loss.item())

    return np.array(scores)


prompt = "The quick brown fox"
N = 100
completions = generate_completions(pi_ref, tokenizer, prompt, n=N)

# Score with pi_star
true_scores = get_log_probs(pi_star, tokenizer, completions)

print(f"Generated {len(completions)} completions.")
print(f"Scores mean: {np.mean(true_scores):.4f}, std: {np.std(true_scores):.4f}")

# 4. Preference Oracle & Graph Estimation


def sample_pairs(n_completions, n_pairs):
    # Randomly select n_pairs pairs of indices (i, j) such that i != j
    pairs = []
    existing = set()
    while len(pairs) < n_pairs:
        idx = np.random.choice(n_completions, 2, replace=False)
        idx = tuple(sorted(idx))
        if idx not in existing:
            existing.add(idx)
            pairs.append(idx)
    return pairs


def label_pairs(pairs, scores):
    # Returns (winner_idx, loser_idx) for each pair
    labeled_pairs = []
    for i, j in pairs:
        if scores[i] > scores[j]:
            labeled_pairs.append((i, j))  # i wins
        else:
            labeled_pairs.append((j, i))  # j wins
    return labeled_pairs


def fit_bradley_terry(n_completions, labeled_pairs):
    # Fit Logistic Regression to learn scores
    # X feature vector: one-hot for winner (+1) and loser (-1)
    # We double the data by adding (loser, winner) pairs with label 0
    # to ensure we have both classes for LogisticRegression.
    
    num_pairs = len(labeled_pairs)
    X = np.zeros((num_pairs * 2, n_completions))
    y = np.zeros(num_pairs * 2)

    for idx, (w, l) in enumerate(labeled_pairs):
        # Sample 1: w beats l (label 1)
        X[idx * 2, w] = 1
        X[idx * 2, l] = -1
        y[idx * 2] = 1
        
        # Sample 2: l loses to w (label 0)
        X[idx * 2 + 1, l] = 1
        X[idx * 2 + 1, w] = -1
        y[idx * 2 + 1] = 0

    # No intercept for BT model (or shared intercept which cancels out)
    lr = LogisticRegression(
        fit_intercept=False, C=1e5
    )  # High C for less regularization
    lr.fit(X, y)

    # The coefficients are the estimated scores (up to a constant shift)
    estimated_scores = lr.coef_[0]
    return estimated_scores


def construct_full_graph(estimated_scores):
    # Create all pairs based on estimated scores
    n = len(estimated_scores)
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            if estimated_scores[i] > estimated_scores[j]:
                pairs.append((i, j))  # i wins
            else:
                pairs.append((j, i))  # j wins
    return pairs


# Parameters
K = 100  # Number of pairs to sample (approx O(N))
sampled_indices = sample_pairs(N, K)
labeled_sample = label_pairs(sampled_indices, true_scores)

# Estimate scores
estimated_scores = fit_bradley_terry(N, labeled_sample)

# Construct full graph
full_graph_pairs = construct_full_graph(estimated_scores)

print(f"Sampled {K} pairs.")
print(f"Constructed full graph with {len(full_graph_pairs)} pairs.")
print(
    f"Correlation between true and estimated scores: {np.corrcoef(true_scores, estimated_scores)[0, 1]:.4f}"
)

# 5. Training Loop


def dpo_loss(policy_model, ref_model, tokenizer, winner_text, loser_text, beta=0.1):
    # Helper to compute log probs
    def get_log_prob_sums(model, text):
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(
            device
        )
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits[:, :-1, :]
        labels = input_ids[:, 1:]

        # Gather log probs of labels
        log_probs = nn.functional.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(dim=-1, index=labels.unsqueeze(-1)).squeeze(
            -1
        )

        # Mask padding
        mask = attention_mask[:, 1:]
        sum_log_probs = (token_log_probs * mask).sum(dim=-1)
        return sum_log_probs

    policy_winner_log_probs = get_log_prob_sums(policy_model, winner_text)
    policy_loser_log_probs = get_log_prob_sums(policy_model, loser_text)

    with torch.no_grad():
        ref_winner_log_probs = get_log_prob_sums(ref_model, winner_text)
        ref_loser_log_probs = get_log_prob_sums(ref_model, loser_text)

    logits = beta * (
        (policy_winner_log_probs - ref_winner_log_probs)
        - (policy_loser_log_probs - ref_loser_log_probs)
    )
    losses = -nn.functional.logsigmoid(logits)
    return losses.mean()


def train_dpo_epoch(
    policy_model,
    ref_model,
    tokenizer,
    pairs,
    completions,
    epochs=10,
    lr=1e-5,
    batch_size=8,
):
    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=lr)
    policy_model.train()

    metrics = {"dist": []}

    train_data = []
    for w_idx, l_idx in pairs:
        train_data.append((completions[w_idx], completions[l_idx]))

    for epoch in tqdm(range(epochs), desc="Epochs"):
        # Shuffle
        np.random.shuffle(train_data)

        epoch_loss = 0
        steps_in_epoch = 0

        # Create batches
        for i in range(0, len(train_data), batch_size):
            batch = train_data[i : i + batch_size]
            winner_texts = [b[0] for b in batch]
            loser_texts = [b[1] for b in batch]

            optimizer.zero_grad()
            loss = dpo_loss(
                policy_model, ref_model, tokenizer, winner_texts, loser_texts
            )
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            steps_in_epoch += 1

        # Evaluation at end of epoch
        dist = 0
        with torch.no_grad():
            for p_train, p_star in zip(policy_model.parameters(), pi_star.parameters()):
                dist += torch.sum((p_train - p_star) ** 2)
        metrics["dist"].append(torch.sqrt(dist).item())

    return metrics


# We need to make tokenizer pad correctly
tokenizer.padding_side = "right"

# Train Baseline
print("Training Baseline DPO...")
metrics_baseline = train_dpo_epoch(
    pi_train_baseline, pi_ref, tokenizer, labeled_sample, completions, epochs=10
)

# Train Graph
print("Training Graph DPO...")
metrics_graph = train_dpo_epoch(
    pi_train_graph, pi_ref, tokenizer, full_graph_pairs, completions, epochs=10
)

# 6. Visualization

plt.figure(figsize=(10, 6))
plt.plot(metrics_baseline["dist"], label="Baseline DPO (Sampled Pairs)", marker="o")
plt.plot(metrics_graph["dist"], label="Graph DPO (Full Graph)", marker="x")
plt.xlabel("Training Steps (Epochs)")
plt.ylabel("Distance to pi_star (L2 norm of weights)")
plt.title("DPO Convergence: Baseline vs Graph")
plt.legend()
plt.grid(True)
plt.savefig("dpo_convergence.png")
print("Plot saved to dpo_convergence.png")


Models initialized: pi_ref, pi_star, pi_train_baseline, pi_train_graph


Scoring: 100%|██████████| 100/100 [00:02<00:00, 39.41it/s]


Generated 100 completions.
Scores mean: -6.1245, std: 0.6955


ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.float64(1.0)